# R-Squared in Regression: What Fit Really Means

<a href="https://www.kaggle.com/code/addarm/r-squared-deepdive" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

![R Squared Deep Dive Banner](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/r_squared_deepdive_banner.png)

Every regression output includes an R-Squared. Every textbook uses it to summarize fit and we all learn to maximize it, which is a bit of a habit, especially when we report it without thinking. Yet R-Squared can mislead even those who understand it well.

R-Squared measures what fraction of in-sample variance the model explains. Add any predictor and it can only rise, and we'll see this in a bit. So packing in more predictors sounds useful, you get a better fit, until you realize it rises for noise and says nothing about causality.

This article builds R-Squared and adjusted R-Squared from scratch, and shows exactly where each one can mislead you.

## Notebook Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(888)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

# Regression Building Blocks

Following our previous article on [linear regressions](https://medium.com/hecatus-research/machine-learning-fundamentals-the-linear-regression-from-intuition-to-geometry-c4cd55e9d2ed) in depth, we will reuse the code to calculate these functions raw.

In [ ]:
def add_intercept(X):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.column_stack([np.ones(X.shape[0]), X])


def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    XtX = X.T @ X
    beta = np.linalg.solve(XtX, X.T @ y)
    y_hat = X @ beta
    resid = y - y_hat
    n = len(y)
    p = X.shape[1] - 1
    sse = float(resid.T @ resid)
    y_bar = float(y.mean())
    sst = float(((y - y_bar) ** 2).sum())
    ssr = float(((y_hat - y_bar) ** 2).sum())
    r2 = 1.0 - sse / sst
    adj_r2 = 1.0 - (sse / (n - p - 1)) / (sst / (n - 1))
    return {
        "beta": beta,
        "y_hat": y_hat,
        "resid": resid,
        "sse": sse,
        "ssr": ssr,
        "sst": sst,
        "r2": r2,
        "adj_r2": adj_r2,
        "n": n,
        "p": p,
    }


def manual_r2(y, y_hat):
    y = np.asarray(y, dtype=float)
    y_hat = np.asarray(y_hat, dtype=float)
    sse = ((y - y_hat) ** 2).sum()
    sst = ((y - y.mean()) ** 2).sum()
    return 1.0 - sse / sst


def train_test_split(X, y, train_frac=0.7, seed=888):
    local_rng = np.random.default_rng(seed)
    n = len(y)
    idx = local_rng.permutation(n)
    n_train = int(train_frac * n)
    return X[idx[:n_train]], X[idx[n_train:]], y[idx[:n_train]], y[idx[n_train:]]


def test_r2_from_train_fit(X_train, y_train, X_test, y_test):
    fit = ols_fit(add_intercept(X_train), y_train)
    y_pred = add_intercept(X_test) @ fit["beta"]
    r2 = manual_r2(y_test, y_pred)
    n = len(y_test)
    p = X_test.shape[1] if X_test.ndim > 1 else 1
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)
    return {"r2": r2, "adj_r2": adj_r2}

# The Data

We design a synthetic dataset where we know the truth: three predictors drive $y$ and eight are pure noise:

$$y_i = 3 + 2x_{i1} - 1.5x_{i2} + 0.75x_{i3} + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0,\,\sigma^2).$$

Because we control the data-generating process, we can watch exactly what R-Squared and adjusted R-Squared do as predictors enter in order, signal first.

In [ ]:
def make_signal_plus_noise_data(n=250, n_signal=3, n_noise=8, sigma=1.5, seed=7):
    local_rng = np.random.default_rng(seed)
    X_signal = local_rng.normal(size=(n, n_signal))
    X_noise = local_rng.normal(size=(n, n_noise))
    beta_signal = np.array([2.0, -1.5, 0.75])[:n_signal]
    y = 3.0 + X_signal @ beta_signal + local_rng.normal(scale=sigma, size=n)
    X_full = np.column_stack([X_signal, X_noise])
    feature_names = [f"x_signal_{j + 1}" for j in range(n_signal)] + [f"x_noise_{j + 1}" for j in range(n_noise)]
    return X_full, y, feature_names


X_full, y, feature_names = make_signal_plus_noise_data()
pd.DataFrame(X_full, columns=feature_names).assign(y=y).head()

# The Sums-of-Squares Identity

Three quantities break down the total variation when an intercept is included:

$$\mathrm{SST} = \sum_{i=1}^n (y_i - \bar{y})^2, \quad \mathrm{SSR} = \sum_{i=1}^n (\hat{y}_i - \bar{y})^2, \quad \mathrm{SSE} = \sum_{i=1}^n (y_i - \hat{y}_i)^2.$$

SST is total variance around the sample mean. SSR is what the model captures. SSE is what's left in the residuals. 

Under OLS with an intercept, the residuals $e = y - \hat{y}$ are orthogonal to the fitted values, so

$$\mathrm{SST} = \mathrm{SSR} + \mathrm{SSE}.$$

The code below verifies this on our data:

In [ ]:
fit_signal = ols_fit(add_intercept(X_full[:, :3]), y)

decomposition_table = pd.DataFrame(
    {
        "quantity": ["SST", "SSR", "SSE", "SSR + SSE"],
        "value": [
            fit_signal["sst"],
            fit_signal["ssr"],
            fit_signal["sse"],
            fit_signal["ssr"] + fit_signal["sse"],
        ],
    }
)
decomposition_table


## R-Squared

The coefficient of determination is:

$$R^2 = \frac{\mathrm{SSR}}{\mathrm{SST}} = 1 - \frac{\mathrm{SSE}}{\mathrm{SST}}.$$

With an intercept, 0 ≤ R-Squared ≤ 1. A higher value means less unexplained variance relative to the mean baseline. 

In simple regression there's a convenient identity: R-Squared = Corr(x, y)² = Corr(y, ŷ)². In multiple regression, only the second equality holds in general.

In [ ]:
x_simple = X_full[:, 0]
fit_simple = ols_fit(add_intercept(x_simple), y)

fig, ax = plt.subplots(figsize=(8, 4))
x_line = np.linspace(x_simple.min(), x_simple.max(), 200)
ax.scatter(x_simple, y, s=16, alpha=0.5, label="data")
ax.plot(x_line, add_intercept(x_line) @ fit_simple["beta"], color="tab:red", linewidth=2, label=f"OLS fit  $R^2$ = {fit_simple['r2']:.3f}")
ax.set_xlabel("x_signal_1")
ax.set_ylabel("y")
ax.set_title("Simple OLS: one predictor")
ax.legend()
plt.show()

# In simple regression R² = Corr(x, y)² = Corr(y, ŷ)²
print(f"R²              = {fit_simple['r2']:.4f}")
print(f"Corr(x, y)²     = {np.corrcoef(x_simple, y)[0, 1] ** 2:.4f}")
print(f"Corr(y, ŷ)²     = {np.corrcoef(y, fit_simple['y_hat'])[0, 1] ** 2:.4f}")

## Why R-Squared Can Increase with Noise

Adding a predictor expands the column space of $X$. OLS projects $y$ onto that space, and adding dimensions can only close the gap between $\hat{y}$ and $y$, so $\mathrm{SSE}$ cannot grow:

$$\mathrm{SSE}_2 \le \mathrm{SSE}_1 \implies R^2_2 \ge R^2_1.$$

A noise predictor picks up a nonzero in-sample coefficient that absorbs some residual variance. R-Squared rises whether or not the variable belongs in the model.

A high R-Squared does not mean you have a good model.

## Adjusted R-Squared

R-Squared has a mechanical flaw: add any predictor and it can only rise. Adjusted R-Squared corrects for this by penalising the number of parameters.

Its formula starts from the same sums of squares, but normalises each by its degrees of freedom rather than its count:

$$\bar{R}^2 = 1 - \frac{\mathrm{SSE}/(n - p - 1)}{\mathrm{SST}/(n - 1)}.$$

The symbols: $n$ is the number of observations, $p$ is the number of predictors (not counting the intercept). Dividing $\mathrm{SSE}$ by $(n - p - 1)$ gives the mean squared residual, an unbiased estimate of the error variance $\sigma^2$. Dividing $\mathrm{SST}$ by $(n - 1)$ gives the sample variance of $y$.

We can rearrange that into the equivalent form you will see most often:

$$\bar{R}^2 = 1 - (1 - R^2)\,\frac{n - 1}{n - p - 1}.$$

The factor $(n-1)/(n-p-1)$ is always greater than 1 when $p > 0$, so adjusted R-Squared is always below raw R-Squared, except at perfect fit where both equal 1. It rises when adding a predictor reduces $\mathrm{SSE}$ enough to offset the denominator shrinking by one: the predictor earns its extra parameter. If the predictor is noise, $\mathrm{SSE}$ barely moves, the penalty bites, and $\bar{R}^2$ falls.

Two things adjusted R-Squared still does not tell you: whether the model is correctly specified, and how it will perform on held-out data. Both of those require separate tests.

# Experiment: Signal Predictors, Then Noise

We fit nested models from one predictor up to eleven. The first three are true signals; the rest are pure noise. The dashed line marks where the true model ends.

In [ ]:
rows = []
for k in range(1, X_full.shape[1] + 1):
    fit_k = ols_fit(add_intercept(X_full[:, :k]), y)
    rows.append(
        {
            "num_predictors": k,
            "last_feature_added": feature_names[k - 1],
            "r2": fit_k["r2"],
            "adjusted_r2": fit_k["adj_r2"],
            "sse": fit_k["sse"],
        }
    )

nested_results = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(nested_results["num_predictors"], nested_results["r2"], marker="o", label="R^2")
ax.plot(nested_results["num_predictors"], nested_results["adjusted_r2"], marker="s", label="Adjusted R^2")
ax.axvline(3, color="black", linestyle="--", linewidth=1, label="True signal dimension")
ax.set_xlabel("Number of predictors")
ax.set_ylabel("Fit score")
ax.set_title("In-sample fit as noise features are added")
ax.legend()
plt.show()

nested_results.loc[:, ["num_predictors", "last_feature_added", "r2", "adjusted_r2"]]


Once the three signal predictors are in, raw R-Squared keeps climbing as we add noise. That's an illusion of fit. Adjusted R-Squared falls back, but only gently: it's penalizing for complexity, not correcting for irrelevance.

Look at the numbers. After predictor 3, raw R-Squared inches from 0.7353 up to 0.7385. Small, but it never drops. Adjusted R-Squared slides from 0.7320 down to 0.7264 over the same range. The penalty is doing something, but it does not scream "stop here." In a real dataset, where you don't know the true model, this subtlety is easy to miss.

# Experiment: In-Sample vs. Out-of-Sample

The adjusted R-Squared above is in-sample. 

The real test is what happens on data the model hasn't seen. We split 70/30, fit on the training set, and evaluate on the hold-out.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_full, y, train_frac=0.7)

comparison_rows = []
for k in range(1, X_full.shape[1] + 1):
    fit_train = ols_fit(add_intercept(X_train[:, :k]), y_train)
    test_scores = test_r2_from_train_fit(X_train[:, :k], y_train, X_test[:, :k], y_test)
    comparison_rows.append(
        {
            "num_predictors": k,
            "train_r2": fit_train["r2"],
            "train_adjusted_r2": fit_train["adj_r2"],
            "test_r2": test_scores["r2"],
            "test_adjusted_r2": test_scores["adj_r2"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(comparison_df["num_predictors"], comparison_df["train_r2"], marker="o", label="Train $R^2$")
ax.plot(comparison_df["num_predictors"], comparison_df["train_adjusted_r2"], marker="s", label="Train adjusted $R^2$")
ax.plot(comparison_df["num_predictors"], comparison_df["test_r2"], marker="^", label="Test $R^2$")
ax.plot(comparison_df["num_predictors"], comparison_df["test_adjusted_r2"], marker="D", linestyle="--", label="Test adjusted $R^2$")
ax.axvline(3, color="black", linestyle="--", linewidth=1, label="True signal dimension")
ax.set_xlabel("Number of predictors")
ax.set_ylabel("Score")
ax.set_title("In-sample vs out-of-sample fit as noise features are added")
ax.legend()
plt.show()

Both metrics peak at three predictors, where test R-Squared approx 0.72 and test adjusted adjusted R-Squared approx 0.71. Past that point they diverge. Test R-Squared slides to 0.69 by eleven predictors. Test adjusted R-Squared falls to 0.63, more than twice the drop. The smaller test set amplifies every unit of degradation: with fewer observations, the penalty factor $(n-1)/(n-p-1)$ bites harder.

# An R-Squared Illusion: Nonlinear Truth, Linear Model

R-Squared measures fit against a linear benchmark. It doesn't check whether the linear specification is correct. 

Here the true conditional mean is quadratic, but we fit a straight line first and see what the score tells us. We generate the data below:

In [ ]:
x = np.linspace(-2.5, 2.5, 220)
y_nonlinear = 1.0 + 2.0 * x + 1.5 * x**2 + rng.normal(scale=1.8, size=len(x))

fit_linear_wrong = ols_fit(add_intercept(x), y_nonlinear)
fit_quadratic = ols_fit(add_intercept(np.column_stack([x, x**2])), y_nonlinear)

pd.DataFrame(
    {
        "model": ["linear only", "linear + quadratic"],
        "r2": [fit_linear_wrong["r2"], fit_quadratic["r2"]],
        "adjusted_r2": [fit_linear_wrong["adj_r2"], fit_quadratic["adj_r2"]],
    }
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x, y_nonlinear, s=18, alpha=0.6, label="data")
ax.plot(x, fit_linear_wrong["y_hat"], color="tab:red", linewidth=2, label="linear fit")
ax.plot(x, fit_quadratic["y_hat"], color="tab:green", linewidth=2, label="quadratic fit")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Misspecified model")
ax.legend()
plt.show()

The linear model scores R-Squared = 0.45. That looks moderate, a bit disappointing maybe, but not catastrophic. The quadratic model scores $0.82$ on the same data.

The data was generated with a quadratic true mean ($1 + 2x + 1.5x^2$), so the linear model is simply the wrong shape. But R-Squared doesn't tell you that. It measures fit against the flat benchmark of predicting $\bar{y}$ for everyone. A straight line through a U-shaped cloud still beats that benchmark by a fair margin, so the score looks reasonable.

So R-Squared = 0.45 could mean 'moderate relationship, some noise,' or it could mean 'completely wrong functional form.' The number looks identical in both cases. You cannot tell from the score alone.

# Negative R-Squared

When you evaluate a model on held-out data, R-Squared can go negative. The formula is the same one we've been using, applied to the hold-out:

$$R^2_{\text{test}} = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y}_{\text{test}})^2}.$$

$\hat{y}_i$ are the model's predictions on the test set, trained on the training set. $\bar{y}_{\text{test}}$ is the mean of the test targets.

When the numerator exceeds the denominator, the ratio exceeds 1 and R-Squared goes below zero: the model's predictions are worse than guessing the test mean.

This happens when the model has overfit: it learned patterns specific to the training set that don't carry over.

# Conclusion

We derived R-Squared and adjusted R-Squared from first principles and ran them through a controlled experiment where we knew the truth.

The core results held up. In OLS with an intercept, R-Squared is mechanically non-decreasing: adding predictors can only help in sample, regardless of whether those predictors belong in the model. Adjusted R-Squared applies a degrees-of-freedom penalty that pushes back against this, but it's still an in-sample number and the pushback is gentle. Out-of-sample R-Squared is the honest test, and it can go negative when a model has overfit badly enough that you'd have been better off predicting the mean.

None of this means R-Squared is useless. It tells you something real about in-sample fit, and that's worth knowing. The problem is overstating what it implies: about causality, about whether the model is correctly specified, or about how it will perform on data it hasn't seen. Know what the number actually measures and it's a fine summary statistic. Use it as a selection criterion or a validation tool and it will mislead you.

See you next time!

# References

1. Lu (2021). *A Rigorous Introduction to Linear Models*. [arXiv:2105.04240](https://arxiv.org/abs/2105.04240)
2. Penn State Eberly College of Science. *Coefficient of Determination, R-Squared*. https://online.stat.psu.edu/stat462/node/95/
3. Penn State Eberly College of Science. *Examples of Using the Coefficient of Determination*. https://online.stat.psu.edu/stat462/node/97/
4. Penn State Eberly College of Science. *Some Cautions on the Interpretation of R-Squared*. https://online.stat.psu.edu/stat462/node/98/
5. Wooldridge, J.M. (2020). *Introductory Econometrics: A Modern Approach*, 7th ed. Cengage. Chapters 2–3 cover R-Squared, adjusted R-Squared, and their interpretation.
6. NIST/SEMATECH. *Measures of Model Fit*. https://www.itl.nist.gov/div898/handbook/pmd/section4/pmd44.htm
7. NIST/SEMATECH. *Residual Plots*. https://www.itl.nist.gov/div898/handbook/pmd/section4/pmd446.htm

## Github

Article is available on [Github](https://github.com/adamd1985/quant_research/blob/main/r_squared_deepdive.ipynb)

## Media

All media used (in the form of code or images) are either solely owned by me, acquired through licensing, or part of the Public Domain and granted use through Creative Commons License.

## CC Licensing and Use

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.